# 🍳 Image-to-Recipe Generation with GEMMA and LoRA (PEFT)
This notebook demonstrates how to fine-tune GEMMA (2B) using LoRA adapters on a Kaggle food image and recipe dataset.

In [ ]:
# 📦 Install required libraries
# If running on Colab:
# !pip install transformers peft accelerate bitsandbytes torchvision pandas


In [1]:
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments, Trainer
from peft import get_peft_model, LoraConfig, TaskType
from torchvision import models, transforms
from torch.utils.data import Dataset
from PIL import Image
import torch, json, os
import pandas as pd


In [2]:
from huggingface_hub import login
import os

hf_token = os.getenv("HF_TOKEN")
if hf_token:
    login(hf_token)
else:
    print("⚠️ Warning: Hugging Face token not found. Please set HF_TOKEN as an env variable.")

⚠️ Warning: Hugging Face token not found. Please set HF_TOKEN as an env variable.


In [15]:
# ✅ Load GEMMA-2B tokenizer & model (use bfloat16 or float16 if on GPU)



model_id = "google/gemma-2b"
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(model_id, torch_dtype=torch.float32, device_map="auto", low_cpu_mem_usage=True)


e:\GeorgiaTech\CS7643\Project\recipe-env\Lib\site-packages\huggingface_hub\file_download.py:896: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [16]:
# ✅ Apply LoRA adapter using PEFT
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.1,
    bias="none",
    task_type=TaskType.CAUSAL_LM
)
model = get_peft_model(model, lora_config)


In [17]:
# Conversion into JSON file
import pandas as pd
import json
import ast  # to safely parse list strings

# Load CSV
df = pd.read_csv("../Data/Kaggle/Kaggle_Food_Recipe.csv")

def is_valid_row(row):
    for val in row:
        if isinstance(val, str) and "#NAME" in val:
            return False
        if val == "[]" or (isinstance(val, list) and len(val) == 0):
            return False
    return True

# Data Analysis
print("Dataset Info:")
df.info()
print("\nMissing Values:")
print(df.isnull().sum())
# Drop rows with any missing values
df = df.dropna()
print(df.isnull().sum())
# Drop duplicates
df = df.drop_duplicates().reset_index(drop=True)

print("\nTop 10 most common ingredients:")
ingredient_counts = df['Cleaned_Ingredients'].str.split(', ').explode().value_counts().head(10)
print(ingredient_counts)
df = df[df.apply(is_valid_row, axis=1)].reset_index(drop=True)




Dataset Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 13501 entries, 0 to 13500
Data columns (total 6 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   Unnamed: 0           13501 non-null  int64 
 1   Title                13496 non-null  object
 2   Ingredients          13501 non-null  object
 3   Instructions         13493 non-null  object
 4   Image_Name           13501 non-null  object
 5   Cleaned_Ingredients  13501 non-null  object
dtypes: int64(1), object(5)
memory usage: 633.0+ KB

Missing Values:
Unnamed: 0             0
Title                  5
Ingredients            0
Instructions           8
Image_Name             0
Cleaned_Ingredients    0
dtype: int64
Unnamed: 0             0
Title                  0
Ingredients            0
Instructions           0
Image_Name             0
Cleaned_Ingredients    0
dtype: int64

Top 10 most common ingredients:
Cleaned_Ingredients
divided'               3810
chopped'  

In [18]:
# Convert ingredients from string to list
def parse_ingredients(val):
    try:
        return ast.literal_eval(val) if isinstance(val, str) else []
    except:
        return []

# Convert to required format
recipes = []
for _, row in df.iterrows():
    recipe = {
        "title": row["Title"],
        "ingredients": parse_ingredients(row["Cleaned_Ingredients"]),
        "instructions": row["Instructions"].strip(),
        "image": row["Image_Name"].strip()
    }
    recipes.append(recipe)

# Save to JSON file
with open("../Data/Kaggle/kaggle_recipes.json", "w") as f:
    json.dump(recipes, f, indent=2)

print(f"Saved {len(recipes)} recipes to subset_recipes.json")

Saved 13457 recipes to subset_recipes.json


In [19]:
# Load Kaggle-style recipes data
with open("../Data/Kaggle/kaggle_recipes.json", "r") as f:
    data = json.load(f)


In [39]:
class RecipeDataset(Dataset):
    def __init__(self, data, image_folder, tokenizer, transform=None, max_length=128):
        self.data = data
        self.image_folder = image_folder
        self.tokenizer = tokenizer
        self.transform = transform or transforms.Compose([
            transforms.Resize((224, 224)),
            transforms.ToTensor()
        ])
        self.max_length = max_length
        # initialize vision encoder ResNet (CNN)
        self.vision_encoder = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
        self.vision_encoder.fc = torch.nn.Identity()
        # Freeze the vision encoder (i.e. no backpropagation and only encoding)
        self.vision_encoder.eval()
        for param in self.vision_encoder.parameters():
            param.requires_grad = False

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        try:
            item = self.data[idx]
            image_filename = os.path.basename(item["image"])
            if not image_filename.lower().endswith((".jpg", ".jpeg", ".png")):
                image_filename += ".jpg"  # default to jpg
            image_path = os.path.join(self.image_folder, image_filename)
            image = Image.open(image_path).convert("RGB")
            image = self.transform(image)
            # Encode image to feature vector
            with torch.no_grad():
                image_embedding = self.vision_encoder(image.unsqueeze(0)).squeeze(0) # shape [512]
            # Prepare text
            prompt = "Generate recipe instructions for this dish:"
            instructions = item["instructions"]

            input_ids = self.tokenizer(prompt, return_tensors="pt", truncation=True, max_length=self.max_length).input_ids.squeeze(0)
            labels = self.tokenizer(instructions, return_tensors="pt", truncation=True, max_length=self.max_length).input_ids.squeeze(0)
            
            # debug line
            # print(f"[{idx}] input_ids: {input_ids.shape}, labels: {labels.shape}, image_embedding: {image_embedding.shape}")
            # print(input_ids)
            print(image_filename)

            return {
                "prompt_input_ids": input_ids,          # shape: [seq_len]
                "labels": labels,                       # shape: [seq_len]
                "image_embedding": image_embedding      # shape: [512]
            }
        except Exception as e:
            print(f"[ERROR] Sample {idx} failed: {e}")
            return None

In [57]:
class ImageToRecipeModel(torch.nn.Module):
    def __init__(self, base_model, embed_dim=512):
        super().__init__()
        self.base_model = base_model
        self.embedding_proj = torch.nn.Linear(embed_dim, base_model.config.hidden_size)

        # Add and freeze the vision encoder
        self.vision_encoder = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
        self.vision_encoder.fc = torch.nn.Identity()
        self.vision_encoder.eval()
        for param in self.vision_encoder.parameters():
            param.requires_grad = False

        # Setup for inference access later
        self.base_model.vision_encoder = self.vision_encoder

    def forward(self, **kwargs):
        # Debug - log incoming keys from Trainer
        # print(f"[forward()] kwargs: {list(kwargs.keys())}")

        # unpack from kwargs
        prompt_input_ids = kwargs["prompt_input_ids"]
        image_embedding = kwargs["image_embedding"]
        labels = kwargs.get("labels", None)

        # Project image embedding to match model hidden size
        projected_image = self.embedding_proj(image_embedding).unsqueeze(1)  # [B, 1, hidden_size]

        # Token embeddings
        input_embeds = self.base_model.get_input_embeddings()(prompt_input_ids)  # [B, T, hidden_size]

        # Concatenate: [image_token | prompt_tokens]
        inputs_embeds = torch.cat([projected_image, input_embeds], dim=1)  # [B, T+1, hidden_size]
       
        # Ensure lengths match
        if labels is not None and labels.shape[1] != inputs_embeds.shape[1]:
            min_len = min(labels.shape[1], inputs_embeds.shape[1])
            labels = labels[:, :min_len]
            inputs_embeds = inputs_embeds[:, :min_len, :]
        
        # Debug print
        # print(f"input_embeds: {input_embeds.shape}, image_embeds: {projected_image.shape}, labels: {labels.shape}")
        
        
        return self.base_model(
            inputs_embeds=inputs_embeds,
            labels=labels
        )
    
    def generate_with_image(self, prompt_input_ids, image_embedding, **generate_kwargs):
        
        # ensure either batched input or single input can all be handled
        if image_embedding.dim() == 1:
            image_embedding = image_embedding.unsqueeze(0)  # [1, hidden]
        if prompt_input_ids.dim() == 1:
            prompt_input_ids = prompt_input_ids.unsqueeze(0)  # [1, T]

        projected_image = self.embedding_proj(image_embedding).unsqueeze(1)  # [B, 1, hidden_size]
        prompt_embeds = self.base_model.get_input_embeddings()(prompt_input_ids) # [B, T, hidden]
        inputs_embeds = torch.cat([projected_image, prompt_embeds], dim=1) # [B, T+1, hidden]
        return self.base_model.generate(inputs_embeds=inputs_embeds, **generate_kwargs)

In [41]:
# custom data_collator 

from torch.nn.utils.rnn import pad_sequence

def image_to_recipe_collator(batch):
    
    # catch and filter any invalid ones
    batch = [item for item in batch if item is not None]

    # Extract each field
    input_ids = [item["prompt_input_ids"] for item in batch]
    labels = [item["labels"] for item in batch]
    image_embs = [item["image_embedding"] for item in batch]

    # Find max length across batch and account for 1 image token (ensure consistency)
    max_prompt_len = max(x.size(0) for x in input_ids)
    max_label_len = max(x.size(0) for x in labels)
    seq_len = max(max_prompt_len, max_label_len)

    # Pad prompt_input_ids and labels to max length in batch
    input_ids_padded = pad_sequence(input_ids, batch_first=True, padding_value=tokenizer.pad_token_id)
    labels_padded = pad_sequence(labels, batch_first=True, padding_value=-100)
    
    # Prepend ignore token to labels for the image embedding
    ignore_label = torch.full((labels_padded.size(0), 1), -100)
    labels = torch.cat([ignore_label, labels_padded], dim=1)

    # Stack image features
    image_embedding = torch.stack(image_embs, dim=0)  # shape: [B, 512]

    # debug
    # print(f"[collator] got keys: {batch[0].keys()}")
    
    return {
        "prompt_input_ids": input_ids_padded,
        "labels": labels,
        "image_embedding": image_embedding
    }

In [42]:
# Prepare data and Trainer
from pathlib import Path
# Get the current working directory (e.g., Project Code/)
cwd = Path.cwd()
image_folder = cwd.parent / "Data" / "Kaggle" / "Food Images"
train_dataset = RecipeDataset(data[:1000], image_folder, tokenizer)
eval_dataset = RecipeDataset(data[1000:1100], image_folder, tokenizer)

training_args = TrainingArguments(
    output_dir="./outputs",
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    evaluation_strategy="steps",
    num_train_epochs=1,
    fp16=False,
    save_steps=100,
    logging_steps=10,
    report_to="none",
    remove_unused_columns=False,
    save_safetensors=False
)


# from accelerate import Accelerator, DataLoaderConfiguration

# # Define DataLoaderConfiguration
# dataloader_config = DataLoaderConfiguration(
#   dispatch_batches=None, 
#   split_batches=False, 
#   even_batches=True, 
#   use_seedable_sampler=True)


# # Initialize Accelerator with DataLoaderConfiguration
# accelerator = Accelerator(dataloader_config=dataloader_config)

trainer = Trainer(
    model=ImageToRecipeModel(base_model=model), 
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=image_to_recipe_collator,
    tokenizer=tokenizer
)


In [43]:
# ✅ Train the model (LoRA tuned)
trainer.train()


  0%|          | 0/250 [00:00<?, ?it/s]

soft-polenta-with-mushrooms-and-spinach.jpg
tomato-and-cheese-cobbler.jpg
barbecued-pork-fried-rice-with-mushrooms-and-extra-ginger.jpg
frozen-avocado-cake-nadine-levy-redzepi.jpg
chicken-thighs-with-tomatoes-and-feta.jpg
grilled-coconut-shrimp-shishito-peppers.jpg
salmon-burgers-with-ginger-and-pickled-cucumbers.jpg
cheddar-potato-soup-with-bacon-15726.jpg
shortcut-puff-pastry.jpg
strawberry-balsamic-shortcakes-with-olive-oil-buttermilk-biscuits.jpg
roasted-carrot-brussels-sprout-and-cranberry-salad.jpg
cbd-caramel-sauce.jpg
crispy-turmeric-and-pepper-baked-chicken-wings.jpg
honeydew-salad-with-ginger-dressing-and-peanuts.jpg
little-gem-salad-with-buttermilk-chaas.jpg
roast-chicken-legs-with-lots-of-garlic.jpg
pistachio-and-pomegranate-meatballs-kufteh-ye-pesteh-o-anar.jpg
maple-roasted-acorn-squash-ina-garten.jpg
pop-it-like-its-hot-homemade-popcorn.jpg
spinach-shiitake-grits-with-sliced-avocado.jpg
drunk-apricot-shito-ghanaian-hot-pepper-sauce.jpg
mango-curry-joe-thottungal.jpg
chic

  0%|          | 0/25 [00:00<?, ?it/s]

leek-top-baking-sheet-hash.jpg
make-ahead-freezer-to-oven-chicken-packets.jpg
spicy-gluten-free-hush-puppies.jpg
chuck-eye-with-carrot-top-salsa-verde.jpg
vegetable-frittata-with-asiago-cheese-4242.jpg
sheet-pan-curry-pork-chops-and-sweet-potatoes.jpg
slow-roast-spiced-lamb-shoulder-with-sumac-onions.jpg
pan-seared-scallops-with-chorizo-and-corn.jpg
baked-cinnamon-toast-with-fruit.jpg
lemon-cake-with-fruit.jpg
slow-cooked-scallions-with-ginger-and-chile.jpg
slow-cooked-eggplant-with-lemon-and-fennel-seeds.jpg
eggplant-with-cashew-butter-and-pickled-peppers.jpg
slow-cooked-bell-peppers-with-bay-leaves-and-oregano.jpg
slow-cooked-winter-squash-with-sage-and-thyme.jpg
roast-chicken-with-bell-peppers-lemon-and-thyme.jpg
grapefruit-orange-crostatas.jpg
slow-cooked-green-beans-with-harissa-and-cumin.jpg
slow-cooked-summer-squash-with-lemon-and-thyme.jpg
slow-cooked-cherry-tomatoes-with-coriander-and-rosemary.jpg
che-fico-chopped-salad.jpg
spaghetti-with-lobster-pomodoro.jpg
ugly-babys-thai-r

  0%|          | 0/25 [00:00<?, ?it/s]

...

  0%|          | 0/25 [00:00<?, ?it/s]

leek-top-baking-sheet-hash.jpg
make-ahead-freezer-to-oven-chicken-packets.jpg
spicy-gluten-free-hush-puppies.jpg
chuck-eye-with-carrot-top-salsa-verde.jpg
vegetable-frittata-with-asiago-cheese-4242.jpg
sheet-pan-curry-pork-chops-and-sweet-potatoes.jpg
slow-roast-spiced-lamb-shoulder-with-sumac-onions.jpg
pan-seared-scallops-with-chorizo-and-corn.jpg
baked-cinnamon-toast-with-fruit.jpg
lemon-cake-with-fruit.jpg
slow-cooked-scallions-with-ginger-and-chile.jpg
slow-cooked-eggplant-with-lemon-and-fennel-seeds.jpg
eggplant-with-cashew-butter-and-pickled-peppers.jpg
slow-cooked-bell-peppers-with-bay-leaves-and-oregano.jpg
slow-cooked-winter-squash-with-sage-and-thyme.jpg
roast-chicken-with-bell-peppers-lemon-and-thyme.jpg
grapefruit-orange-crostatas.jpg
slow-cooked-green-beans-with-harissa-and-cumin.jpg
slow-cooked-summer-squash-with-lemon-and-thyme.jpg
slow-cooked-cherry-tomatoes-with-coriander-and-rosemary.jpg
che-fico-chopped-salad.jpg
spaghetti-with-lobster-pomodoro.jpg
ugly-babys-thai-r

  0%|          | 0/25 [00:00<?, ?it/s]

leek-top-baking-sheet-hash.jpg
make-ahead-freezer-to-oven-chicken-packets.jpg
spicy-gluten-free-hush-puppies.jpg
chuck-eye-with-carrot-top-salsa-verde.jpg
vegetable-frittata-with-asiago-cheese-4242.jpg
sheet-pan-curry-pork-chops-and-sweet-potatoes.jpg
slow-roast-spiced-lamb-shoulder-with-sumac-onions.jpg
pan-seared-scallops-with-chorizo-and-corn.jpg
baked-cinnamon-toast-with-fruit.jpg
lemon-cake-with-fruit.jpg
slow-cooked-scallions-with-ginger-and-chile.jpg
slow-cooked-eggplant-with-lemon-and-fennel-seeds.jpg
eggplant-with-cashew-butter-and-pickled-peppers.jpg
slow-cooked-bell-peppers-with-bay-leaves-and-oregano.jpg
slow-cooked-winter-squash-with-sage-and-thyme.jpg
roast-chicken-with-bell-peppers-lemon-and-thyme.jpg
grapefruit-orange-crostatas.jpg
slow-cooked-green-beans-with-harissa-and-cumin.jpg
slow-cooked-summer-squash-with-lemon-and-thyme.jpg
slow-cooked-cherry-tomatoes-with-coriander-and-rosemary.jpg
che-fico-chopped-salad.jpg
spaghetti-with-lobster-pomodoro.jpg
ugly-babys-thai-r

TrainOutput(global_step=250, training_loss=5.7797215423583985, metrics={'train_runtime': 1763.9686, 'train_samples_per_second': 0.567, 'train_steps_per_second': 0.142, 'train_loss': 5.7797215423583985, 'epoch': 1.0})

In [69]:
import gc, torch
gc.collect()
torch.cuda.empty_cache()

In [60]:
from peft import PeftModel
import os

# Define save path
save_dir = "./lora_image_to_recipe"
os.makedirs(save_dir, exist_ok=True)

final_model = PeftModel(model, lora_config)

# Save tokenizer
tokenizer.save_pretrained(save_dir)

# Save LoRA base model (Gemma + LoRA adapters)
final_model.save_pretrained(save_dir)

print("Model and tokenizer saved.")

Model and tokenizer saved.


In [ ]:
# Reload from saved trained model

# from transformers import AutoTokenizer, AutoModelForCausalLM
# from peft import PeftModel
# from torchvision import models
# import torch

# 1. Load tokenizer
tokenizer = AutoTokenizer.from_pretrained("./lora_image_to_recipe")

# 2. Load base model **without device_map first**
base_model = AutoModelForCausalLM.from_pretrained(
    "google/gemma-2b",
    torch_dtype=torch.float32  # or bfloat16 if supported
)

# 3. Load LoRA adapter
lora_model = PeftModel.from_pretrained(
    base_model,
    "./lora_image_to_recipe",
    local_files_only=True
)

# # 4. Move model to device and/or use Accelerate
lora_model = lora_model.to("cuda").eval()

# 5. Wrap in your custom vision+text model
model = ImageToRecipeModel(base_model=lora_model)
model = model.eval()

# # Wrap into custom image-to-recipe model
# wrapper_model = ImageToRecipeModel(base_model=lora_model)

# # Attach vision encoder
# vision_encoder = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
# vision_encoder.fc = torch.nn.Identity()
# vision_encoder.eval()
# for p in vision_encoder.parameters():
#     p.requires_grad = False

# wrapper_model.vision_encoder = vision_encoder
# wrapper_model.to("cuda").eval()

print("Model and tokenizer reloaded.")

e:\GeorgiaTech\CS7643\Project\recipe-env\Lib\site-packages\huggingface_hub\file_download.py:896: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [65]:
# Set up for inference later
@torch.no_grad()
def generate_recipe_from_image(image_path, model, tokenizer, device="cuda", prompt="Generate recipe instructions for this dish:"):
    # 1. Load and preprocess image
    transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor()
    ])
    image = Image.open(image_path).convert("RGB")
    image_tensor = transform(image).unsqueeze(0).to(device)

    # 2. Encode image using the model's vision encoder
    model.eval()
    vision_encoder = model.vision_encoder  # directly attached to wrapper
    image_embedding = vision_encoder(image_tensor).squeeze(0)  # shape: [512]

    # 3. Tokenize prompt
    prompt_input_ids = tokenizer(prompt, return_tensors="pt").input_ids.to(device)

    # 4. Generate recipe
    output = model.generate_with_image(
        prompt_input_ids=prompt_input_ids,
        image_embedding=image_embedding,
        max_new_tokens=200,
        do_sample=True,
        top_k=50,
        top_p=0.95,
        temperature=0.7
    )

    # 5. Decode output
    return tokenizer.decode(output[0], skip_special_tokens=True)

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
# device = "cpu"
# move vision encoder to same device as inputs
model.vision_encoder = model.vision_encoder.to(device)

image_path = cwd.parent / "Data" / "Kaggle" / "Food Images"
output_text = generate_recipe_from_image(
    image_path=image_path / "tomato-and-cheese-cobbler.jpg",
    model=model,
    tokenizer=tokenizer,
    device=device
)
print(output_text)

KeyboardInterrupt: 